# 05: Experimento v4: Recalibração trimestral

**O que muda nesta versão:** Mesmos dados e variáveis da v3. Na divisão temporal, os modelos são retreinados a cada trimestre do teste com tudo o que já era conhecido no início do trimestre (treino + desfechos dos trimestres anteriores; janela expansiva). Hiperparâmetros escolhidos uma vez, só no treino, como na v3. A divisão agrupada não tem ordem no tempo e fica igual à v3.

**Versão anterior:** `05_experimento_v3.ipynb`

## Protocolo (igual em todas as versões)

- **Duas divisões 70/30**, as mesmas em todas as versões, para as comparações serem pareadas:
  - **agrupada:** pacientes sorteados até o teste reunir 30% dos registros; nenhum paciente nos dois lados;
  - **temporal:** os 70% de registros mais antigos para treino, e os 30% mais recentes para teste (corte em set/2019).
    `periodo` fica de fora, porque é constante no treino.
- **Quatro modelos:**
  - Rede Bayesiana reduzida ao Manto de Markov, com critério BIC e com AIC;
  - **regressão logística** (one-hot + padronização; regularização escolhida na validação interna);
  - **LightGBM** (30 tentativas do Optuna na validação interna).
- **Validação interna só com os 70% de treino:** CV agrupada (3 folds) na divisão agrupada; os últimos 25% do
  período de treino na divisão temporal.
- **Incerteza:** intervalos de 95% por **bootstrap reamostrando pacientes** (1.000 reamostragens) e diferenças
  pareadas entre modelos nas mesmas reamostragens.

Código: `src/noshow/registry.py` (entrada `"v4"`), resultados em `resultados/experimentos/v4/`.

In [ ]:
import json
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from noshow.benchmark import grouped_bootstrap, paired_difference, to_lgbm
from noshow.registry import EXPERIMENTOS, MODELOS, RESULTADOS, load, load_bases, run, splits

warnings.filterwarnings("ignore")
assert os.environ.get("PYTHONHASHSEED") == "0", "Use o kernel do projeto (PYTHONHASHSEED=0); ver README."

VERSAO = "v4"
ANTERIOR = 'v3'
RECALCULAR = False  # True refaz o experimento (~5 min); False reaproveita resultados/experimentos/v4/

if RECALCULAR or not (RESULTADOS / VERSAO / "predicoes.parquet").exists():  # predições não são versionadas
    M, P = run(VERSAO)
else:
    M, P = load(VERSAO)

BLUE, ORANGE, GRAY, INK, GREEN = "#2a78d6", "#eb6834", "#8a8984", "#52514e", "#1baf7a"
COR = {"RB (BIC)": ORANGE, "RB (AIC)": "#eda100", "Regressão logística": BLUE, "LightGBM": GREEN}
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": GRAY, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
})
print(f"Experimento {VERSAO}: {EXPERIMENTOS[VERSAO]['titulo']}")

## 1. Resultados no teste (30%)

In [ ]:
tabela = M.set_index(["divisao", "modelo"]).loc[[(d, m) for d in ["agrupada", "temporal"] for m in MODELOS]]
tabela[["n_variaveis", "roc_auc", "pr_auc", "brier", "p_media", "auc_interna", "manto"]].style.format(
    {"roc_auc": "{:.4f}", "pr_auc": "{:.4f}", "brier": "{:.4f}", "p_media": "{:.1%}", "auc_interna": "{:.4f}"},
    na_rep="—")

## 2. Intervalos de confiança (bootstrap por paciente)

Uma única divisão 70/30 dá um único número por modelo. O bootstrap estima o quanto esse número variaria com outra
amostra de pacientes: reamostra os pacientes do teste com reposição e recalcula as métricas 1.000 vezes.

In [ ]:
BOOT = {}
for divisao in ["agrupada", "temporal"]:
    sub = P[P.divisao == divisao]
    base_ = sub[sub.modelo == MODELOS[0]].sort_values("linha")
    preds = {m: sub[sub.modelo == m].sort_values("linha").p.to_numpy() for m in MODELOS}
    BOOT[divisao] = grouped_bootstrap(base_.y.to_numpy(), preds, base_.grupo.to_numpy(), n_boot=1000)

ic = pd.DataFrame([
    {"divisao": d, "modelo": m, "métrica": met, "média": BOOT[d][f"{m}|{met}"].mean(),
     "ic_inf": np.quantile(BOOT[d][f"{m}|{met}"], 0.025), "ic_sup": np.quantile(BOOT[d][f"{m}|{met}"], 0.975)}
    for d in BOOT for m in MODELOS for met in ["roc_auc", "pr_auc"]])

fig, axes = plt.subplots(2, 2, figsize=(13, 5.5), sharey=True)
for (i, divisao), (j, met) in [(a, b) for a in enumerate(["agrupada", "temporal"]) for b in enumerate(["roc_auc", "pr_auc"])]:
    ax = axes[i, j]
    for k, m in enumerate(MODELOS[::-1]):
        r = ic[(ic.divisao == divisao) & (ic.modelo == m) & (ic["métrica"] == met)].iloc[0]
        ax.plot([r.ic_inf, r.ic_sup], [k, k], color=COR[m], lw=3, solid_capstyle="round")
        ax.plot(r["média"], k, "o", color=COR[m], ms=8, mec="white", mew=2)
        ax.text(r.ic_sup, k, f"  {r['média']:.3f}", va="center", fontsize=8, color=INK)
    ax.set_yticks(range(len(MODELOS)), MODELOS[::-1])
    ax.set_title(f"{divisao} — {'ROC-AUC' if met == 'roc_auc' else 'PR-AUC'}")
    if met == "roc_auc":
        ax.axvline(0.5, color=GRAY, ls="--", lw=1)
    ax.grid(axis="y", visible=False)
fig.suptitle("Estimativa e IC 95% no teste", x=0.01, ha="left", fontweight="bold")
plt.tight_layout(); plt.show()

**Comparação com a Rede Bayesiana (BIC)**, o modelo mais parcimonioso do TCC: diferença pareada de ROC-AUC (outro
modelo − RB BIC), com IC 95%.

In [ ]:
comp = pd.DataFrame([{"divisao": d, "modelo": m, **paired_difference(BOOT[d], m, "RB (BIC)", "roc_auc")}
                     for d in BOOT for m in MODELOS[1:]]).set_index(["divisao", "modelo"])
comp.style.format({"diferença": "{:+.4f}", "ic_inf": "{:+.4f}", "ic_sup": "{:+.4f}", "p": "{:.3f}"}) \
    .map(lambda v: "font-weight: bold" if v < 0.05 else "", subset=["p"])

## 3. Comparação com a versão anterior (v3)

O mesmo modelo na mesma divisão e com **os mesmos registros de teste**: a diferença mede só o efeito da mudança
desta versão.

In [ ]:
_, P_ant = load(ANTERIOR)
linhas = []
for divisao in ["agrupada", "temporal"]:
    for m in MODELOS:
        a = P[(P.divisao == divisao) & (P.modelo == m)].sort_values("linha")
        b = P_ant[(P_ant.divisao == divisao) & (P_ant.modelo == m)].sort_values("linha")
        assert (a.linha.to_numpy() == b.linha.to_numpy()).all()
        boot = grouped_bootstrap(a.y.to_numpy(), {"atual": a.p.to_numpy(), "anterior": b.p.to_numpy()},
                                 a.grupo.to_numpy(), n_boot=1000)
        for met in ["roc_auc", "pr_auc"]:
            linhas.append({"divisao": divisao, "modelo": m, "métrica": met,
                           "anterior": boot[f"anterior|{met}"].mean(), "atual": boot[f"atual|{met}"].mean(),
                           **paired_difference(boot, "atual", "anterior", met)})
evolucao = pd.DataFrame(linhas).set_index(["divisao", "modelo", "métrica"])
evolucao.style.format({"anterior": "{:.4f}", "atual": "{:.4f}", "diferença": "{:+.4f}", "ic_inf": "{:+.4f}",
                       "ic_sup": "{:+.4f}", "p": "{:.3f}"}) \
    .map(lambda v: "font-weight: bold" if v < 0.05 else "", subset=["p"])

## 4. Evolução das versões registradas

Melhor ROC-AUC de cada versão já executada (todas com as mesmas divisões).

In [ ]:
todas = pd.concat([load(v)[0] for v in EXPERIMENTOS if (RESULTADOS / v / "metricas.csv").exists()
                   and list(EXPERIMENTOS).index(v) <= list(EXPERIMENTOS).index(VERSAO)])
evol = todas.pivot_table(index=["divisao", "modelo"], columns="versao", values="roc_auc")
evol.style.format("{:.4f}").highlight_max(axis=1, color="#d6e6f8")

## 5. O que os modelos usam

- **Rede Bayesiana:** o Manto de Markov (coluna `manto` na Seção 1) é o modelo inteiro.
- **LightGBM:** importância média das variáveis pelo **SHAP** (contribuição média absoluta para o log-odds),
  reajustado no treino com os hiperparâmetros escolhidos pelo Optuna.

In [ ]:
import lightgbm as lgb
import shap

cfg = EXPERIMENTOS[VERSAO]
bases = load_bases()
base = bases[cfg["base"]]
parametros = json.loads((RESULTADOS / VERSAO / "parametros.json").read_text())["modelos"]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (divisao, (tr, te)) in zip(axes, splits(bases).items()):
    cat = [c for c in cfg["categoricas"] if not (divisao == "temporal" and c == "periodo")]
    cols = cat + cfg["continuas"]
    Xtr, categorias = to_lgbm(base[cols].iloc[tr], cat)
    hp = {k: v for k, v in parametros[f"{divisao}|LightGBM"].items() if k != "auc_interna"}
    modelo = lgb.LGBMClassifier(**hp, random_state=42, verbose=-1).fit(Xtr, base.no_show.to_numpy()[tr])
    Xte = to_lgbm(base[cols].iloc[te], cat, categorias)[0]
    valores = shap.TreeExplainer(modelo).shap_values(Xte.sample(3000, random_state=0))
    valores = valores[1] if isinstance(valores, list) else valores
    imp = pd.Series(np.abs(valores).mean(axis=0), index=cols).sort_values().tail(12)
    ax.barh(imp.index, imp.values, color=GREEN, height=0.6)
    ax.set_title(f"LightGBM — {divisao}: |SHAP| médio"); ax.grid(axis="y", visible=False)
    ax.tick_params(axis="y", labelsize=8)
plt.tight_layout(); plt.show()

## 6. Registro do experimento

**Data:** 01/11/2026 · **Variáveis:** 33 (iguais à v3) · **ID:** aproximado · **Mudança:** recalibração trimestral
(janela expansiva) na divisão temporal; 11 retreinos de 2019-T3 a 2022-T1

**Verificação:** a divisão agrupada, que não mudou, reproduz a v3 **exatamente** (diferença 0,0000 nos quatro modelos).

| Divisão temporal | v3 → v4 (ROC-AUC) | Diferença [IC 95%] | PR-AUC v3 → v4 |
|---|---|---|---|
| **LightGBM** | 0,552 → **0,568** | **+0,016 [+0,005; +0,028]** | 0,171 → **0,182** (+0,011, *p* = 0,02) |
| **RB (BIC)** | 0,534 → 0,551 | **+0,017 [+0,006; +0,028]** | 0,150 → 0,161 (+0,011, *p* < 0,001) |
| RB (AIC) | 0,506 → 0,527 | +0,021 [+0,007; +0,035] | 0,144 → 0,151 |
| Regressão logística | 0,555 → 0,552 | −0,003 (n.s.) | 0,163 → 0,168 (n.s.) |

**Leitura:**
- **A recalibração melhora três dos quatro modelos**, com ganho pequeno, mas consistente e significativo: cerca de
  +0,02 de ROC-AUC e +0,01 de PR-AUC. O LightGBM passa a ser o melhor modelo na divisão temporal (0,568).
- **O ganho vem do nível de risco, e não de uma ordenação melhor dentro de cada período.** A taxa de faltas sobe de
  11% (2019) para 17% (2022). Retreinando, o modelo acompanha parte dessa subida, e a AUC calculada sobre todo o
  teste melhora porque os períodos de risco maior recebem probabilidades maiores.
- **Mas a correção é parcial.** Risco médio previsto × real em 2022: LightGBM 12,3% × 17,2% (v3: 9,1%) e
  RB (BIC) 12,8% × 17,2% (v3: 11,6%). A janela expansiva mantém os anos antigos, de taxa baixa, no treino e dilui a
  atualização.
- **O Manto da RB (BIC) muda ao longo do tempo:** `taxa_faltas_90d` até 2021-T2 e `taxa_faltas_anteriores` a partir
  de 2021-T3. Com mais histórico acumulado, a taxa de longo prazo volta a ser a melhor resumidora. O modelo
  continua com uma única variável em todos os 11 retreinos.
- **Para o serviço:** reestimar o modelo periodicamente é barato (a RB é uma contagem; o LightGBM leva segundos com
  hiperparâmetros fixos) e deve fazer parte da operação.

**Balanço das quatro versões (ROC-AUC, melhor modelo):**

| | Agrupada | Temporal |
|---|---|---|
| v1: linha de base | LightGBM 0,678 | RB (BIC) 0,559 |
| v2: variáveis novas | LightGBM 0,684 | RB (BIC) 0,546 |
| v3: ID aproximado | LightGBM 0,678 | logística 0,555 |
| v4: recalibração | LightGBM 0,678 | **LightGBM 0,568** |

**Próximo experimento sugerido (v5):** **janela deslizante** (retreinar só com os últimos 12 meses), para acompanhar
a deriva com mais rapidez. O risco é ter menos dados: o período de 2020 tem pouquíssimos registros.